In [1]:
# guess_country.py
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from collections import Counter

In [ ]:
# Init Chroma
chroma_client = chromadb.PersistentClient(".chroma_db")
collection = chroma_client.get_collection(name="wikipedia")

In [7]:
embedder = SentenceTransformer("intfloat/e5-base-v2")
flan = pipeline("text2text-generation", model="google/flan-t5-base")

Device set to use mps:0


In [22]:
statement = "This country is rich in oil resources."

In [30]:
# Get embeddings corresponding to statement
top_k = 5
results = collection.query(
    query_texts=[ statement ],
    n_results=top_k
)
print(results)

{'ids': [['Malaysia_26934', 'Algeria_762', 'Norway_33459', 'Venezuela_48685', 'Kyrgyzstan_24156']], 'embeddings': None, 'documents': [["one of the world's largest producers of palm oil.", '=== Oil and natural resources ===', '=== Resources ===\n\n\n==== Oil industry ====', '=== Petroleum and other resources ===', "Kumtor Gold Mine and other regions. The country's plentiful water resources and mountainous terrain enable it to produce and export large quantities of hydroelectric energy."]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'country': 'Malaysia'}, {'country': 'Algeria'}, {'country': 'Norway'}, {'country': 'Venezuela'}, {'country': 'Kyrgyzstan'}]], 'distances': [[0.26348215341567993, 0.27896198630332947, 0.28568583726882935, 0.28979748487472534, 0.29985153675079346]]}


In [ ]:
# print(results['metadatas'])

[[{'country': 'Malaysia'}, {'country': 'Algeria'}, {'country': 'Norway'}, {'country': 'Venezuela'}, {'country': 'Kyrgyzstan'}]]


In [34]:
countries = [ entry['country'] for entry in results['metadatas'][0] ]
print(countries)

['Malaysia', 'Algeria', 'Norway', 'Venezuela', 'Kyrgyzstan']


In [38]:
# Re-query DB filtering country
results = {}
for country in countries:
    country_results = collection.query(
        query_texts=[ statement ],
        n_results=top_k,
        where={'country': country}
    )
    country_texts = country_results['documents'][0]
    results[country] = country_texts
print(results)

{'Malaysia': ["one of the world's largest producers of palm oil.", "The country's economy has traditionally been driven by its natural resources but is expanding into commerce, tourism, and medical tourism. The country has a newly industrialised market economy, which is relatively open and state-oriented. The country is a founding member of the Organisation of Islamic Cooperation (OIC), the East Asia Summit (EAS), and the Association of Southeast Asian Nations (ASEAN), and a member of the Non-Aligned Movement (NAM), the Commonwealth, and the Asia-Pacific", 'and Herzegovina, Somalia, Kosovo, East Timor, and Lebanon.', "International trade, facilitated by the shipping route in adjacent Strait of Malacca, and manufacturing are the key sectors. Malaysia is an exporter of natural and agricultural resources, and petroleum is a major export. Malaysia has once been the largest producer of tin, rubber and palm oil in the world. Manufacturing has a large influence in the country's economy, altho

In [24]:
# Combine the contexts into a single string, including country names
context = ""
for id in results['ids'][0]:
    text = collection.get(id)['documents'][0]
    country = collection.get(id)['metadatas'][0]['country']
    chunk = f"{id}. {country}: {text}\n"
    # print(chunk)
    context = context + chunk
print(context)

Malaysia_26934. Malaysia: one of the world's largest producers of palm oil.
Algeria_762. Algeria: === Oil and natural resources ===
Norway_33459. Norway: === Resources ===


==== Oil industry ====
Venezuela_48685. Venezuela: === Petroleum and other resources ===
Kyrgyzstan_24156. Kyrgyzstan: Kumtor Gold Mine and other regions. The country's plentiful water resources and mountainous terrain enable it to produce and export large quantities of hydroelectric energy.



In [ ]:
# Simple Q&A prompt
prompt = f"Answer based on context:\n\n{context}\n\n{statement}"

print(prompt)

In [10]:
# Keep track of answers and scores
description = ""
scores = Counter()

def generate_question(description):
    prompt = f"You are playing 20 Questions to guess a country. Based on what we know so far: {description}\nSuggest one open-ended question that will help identify the country."
    q = flan(prompt, max_length=50, do_sample=False)[0]['generated_text']
    return q.strip()

def update_scores(description):
    query_vec = embedder.encode(description).tolist()
    results = collection.query(query_embeddings=[query_vec], n_results=5)
    for meta in results['metadatas'][0]:
        country = meta['country']
        scores[country] += 1


In [11]:
# Gameplay loop
for turn in range(5):  # Fixed number of questions
    question = generate_question(description) if description else "Tell me something about the country you're thinking of."
    answer = input(f"{turn+1}. {question} ")
    description += " " + answer
    update_scores(description)

# Final guess
best_match = scores.most_common(1)[0][0]
print(f"My guess is: {best_match}!")


Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


My guess is: Singapore!
